# Pandas Data Engineering

## Imports and Data Setup

In [1]:
import pandas as pd
import numpy as np

left = pd.DataFrame({
    "sample_id":[101,102,103],
    "sample_name":["S1","S2","S3"],
    "sample_date":["2021-01-01","2021-01-02","2021-01-03"],
    "wet_mass_g":[250.1,240.5,np.nan],
    "report_date":["2021-02-04","2021-02-01","2021-02-10"],
    "project":['A','A','A']
})

right = pd.DataFrame({
    "sample_id":[101,102,104],
    "dry_mass_g":[220.0,210.2,199.9],
    "tech":['A','B','C']
})

## Data Merge  (SQL - style Join)

In [2]:
merged_inner = left.merge(right,on="sample_id",how="inner")
merged_right = left.merge(right,on="sample_id",how="right")
merged_left = left.merge(right,on="sample_id",how="left")

In [3]:
print(merged_inner)

   sample_id sample_name sample_date  wet_mass_g report_date project  \
0        101          S1  2021-01-01       250.1  2021-02-04       A   
1        102          S2  2021-01-02       240.5  2021-02-01       A   

   dry_mass_g tech  
0       220.0    A  
1       210.2    B  


In [4]:
print(merged_right)

   sample_id sample_name sample_date  wet_mass_g report_date project  \
0        101          S1  2021-01-01       250.1  2021-02-04       A   
1        102          S2  2021-01-02       240.5  2021-02-01       A   
2        104         NaN         NaN         NaN         NaN     NaN   

   dry_mass_g tech  
0       220.0    A  
1       210.2    B  
2       199.9    C  


In [5]:
print(merged_left)

   sample_id sample_name sample_date  wet_mass_g report_date project  \
0        101          S1  2021-01-01       250.1  2021-02-04       A   
1        102          S2  2021-01-02       240.5  2021-02-01       A   
2        103          S3  2021-01-03         NaN  2021-02-10       A   

   dry_mass_g tech  
0       220.0    A  
1       210.2    B  
2         NaN  NaN  


## Data Blend (Combines data sources for analysis)

In [6]:
blended = left.merge(right,on="sample_id",how="outer")
blended = blended.fillna('Unknown')  #Fills in gaps with 'Unknown'
blended

,sample_id,sample_name,sample_date,wet_mass_g,report_date,project,dry_mass_g,tech
0,101,S1,2021-01-01,250.1,2021-02-04,A,220.0,A
1,102,S2,2021-01-02,240.5,2021-02-01,A,210.2,B
2,103,S3,2021-01-03,Unknown,2021-02-10,A,Unknown,Unknown
3,104,Unknown,Unknown,Unknown,Unknown,Unknown,199.9,C


## Appending (add new rows)

In [7]:
new_rows = pd.DataFrame({'sample_id':[105],
                         'sample_name':['S6'],
                         'sample_date':['2021-01-04'],
                         'wet_mass_g':[200.0],
                         'report_date':['2021-02-05']})
left_append = pd.concat([left,new_rows],ignore_index=True)
left_append

,sample_id,sample_name,sample_date,wet_mass_g,report_date,project
0,101,S1,2021-01-01,250.1,2021-02-04,A
1,102,S2,2021-01-02,240.5,2021-02-01,A
2,103,S3,2021-01-03,NaN,2021-02-10,A
3,105,S6,2021-01-04,200.0,2021-02-05,NaN


## Concatenation (stack rows or columns)

### Vertical (stack rows)

In [8]:
stacked = pd.concat([left,left],axis=0,ignore_index=True)
stacked

,sample_id,sample_name,sample_date,wet_mass_g,report_date,project
0,101,S1,2021-01-01,250.1,2021-02-04,A
1,102,S2,2021-01-02,240.5,2021-02-01,A
2,103,S3,2021-01-03,NaN,2021-02-10,A
3,101,S1,2021-01-01,250.1,2021-02-04,A
4,102,S2,2021-01-02,240.5,2021-02-01,A
5,103,S3,2021-01-03,NaN,2021-02-10,A


### Horizontal (stack columns; aligns by index)

In [9]:
wide = pd.concat([left.reset_index(drop=True),left.reset_index(drop=True)],axis=1)
wide

,sample_id,sample_name,sample_date,wet_mass_g,report_date,project,sample_id,sample_name,sample_date,wet_mass_g,report_date,project
0,101,S1,2021-01-01,250.1,2021-02-04,A,101,S1,2021-01-01,250.1,2021-02-04,A
1,102,S2,2021-01-02,240.5,2021-02-01,A,102,S2,2021-01-02,240.5,2021-02-01,A
2,103,S3,2021-01-03,NaN,2021-02-10,A,103,S3,2021-01-03,NaN,2021-02-10,A


## Imputation (fills missing values)

### A - Fill missing values with constants / per-column values: fillna

In [10]:
imputed = left.copy()
imputed['wet_mass_g'] = imputed['wet_mass_g'].fillna(imputed['wet_mass_g'].mean())
imputed

,sample_id,sample_name,sample_date,wet_mass_g,report_date,project
0,101,S1,2021-01-01,250.1,2021-02-04,A
1,102,S2,2021-01-02,240.5,2021-02-01,A
2,103,S3,2021-01-03,245.3,2021-02-10,A


### B - Interpolate numeric gaps: interpolate

In [11]:
series = pd.Series([1,np.nan,np.nan,4])
series.interpolate(methd='linear')

0    1.0
1    2.0
2    3.0
3    4.0
dtype: float64

## Reduction (reduce columns/rows)

### Drop columns

In [12]:
reduced_cols = left.drop(columns=['sample_name','sample_date'])
reduced_cols

,sample_id,wet_mass_g,report_date,project
0,101,250.1,2021-02-04,A
1,102,240.5,2021-02-01,A
2,103,NaN,2021-02-10,A


### Dropped rows

In [13]:
reduced_rows =left.drop(index=[0])  #Drop first row
reduced_rows

,sample_id,sample_name,sample_date,wet_mass_g,report_date,project
1,102,S2,2021-01-02,240.5,2021-02-01,A
2,103,S3,2021-01-03,NaN,2021-02-10,A


### Filter rows

In [14]:
filtered = left[left['wet_mass_g'].notna()] #Keep only rows with a value
filtered

,sample_id,sample_name,sample_date,wet_mass_g,report_date,project
0,101,S1,2021-01-01,250.1,2021-02-04,A
1,102,S2,2021-01-02,240.5,2021-02-01,A


## Aggregation (group + summarize)


In [15]:
agg1 = left.groupby('project')['wet_mass_g'].mean()
agg1

project
A    245.3
Name: wet_mass_g, dtype: float64

In [16]:
agg2 = left.groupby('project').agg(
    n_samples = ('sample_id','count'),
    avg_wet_mass_g = ('wet_mass_g','mean'),
    min_wet_mass_g = ('wet_mass_g','min'),
    max_wet_mass_g = ('wet_mass_g','max'),
    )
agg2

,n_samples,avg_wet_mass_g,min_wet_mass_g,max_wet_mass_g
project,,,,
A,3,245.3,240.5,250.1


## Transposing (swap rows / columns)

### Using.T

In [17]:
t1 = left.T
t1

,0,1,2
sample_id,101,102,103
sample_name,S1,S2,S3
sample_date,2021-01-01,2021-01-02,2021-01-03
wet_mass_g,250.1,240.5,NaN
report_date,2021-02-04,2021-02-01,2021-02-10
project,A,A,A


### Using .transpose()

In [18]:
t2 = left.transpose()
t2

,0,1,2
sample_id,101,102,103
sample_name,S1,S2,S3
sample_date,2021-01-01,2021-01-02,2021-01-03
wet_mass_g,250.1,240.5,NaN
report_date,2021-02-04,2021-02-01,2021-02-10
project,A,A,A


## Normalization

### Normalize nested JSON -> flat table (json_normalize)

In [19]:
data = [
    {'sample_id':1, "loc":{'x':10,'y':20},'meta':{'size':'A'}},
    {'sample_id':2, "loc":{'x':15,'y':25},'meta':{'size':'B'}},
]
flat = pd.json_normalize(data,sep='.')
flat

,sample_id,loc.x,loc.y,meta.size
0,1,10,20,A
1,2,15,25,B


# Normalize numeric values (feature scaling)

In [21]:
df=left.copy()
col = "wet_mass_g"
nor = df[col + "_minmax"] = (df[col]-df[col].min()) / (df[col].max() - df[col].min())
nor

0    1.0
1    0.0
2    NaN
Name: wet_mass_g, dtype: float64

### Z-score (mean 0, std 1)

In [23]:
z= df[col + "_z"] = (df[col] - df[col].mean()) / df[col].std()
z

0    0.707107
1   -0.707107
2         NaN
Name: wet_mass_g, dtype: float64